<a href="https://colab.research.google.com/github/Seif-Abouelkhair/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-Abouelkhair/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use a shallow Decision Tree classifier because the target is binary: whether a content item experienced an impressions decline of more than 20%.

A Decision Tree fits this lane because it can model simple nonlinear relationships while remaining easy to inspect. I can print the learned decision rules and see which features are used to separate declining from non-declining content.

I will keep the tree shallow rather than optimizing for complexity. The goal is to test whether the features contain useful predictive signal and whether that signal generalizes across clients, not simply to maximize a score on one split.

I will compare the model against the Week-4 baseline using the same target and evaluation metric, while using a client-grouped split to make the evaluation more realistic.


I will use a grouped train/test split by `client_hash_id`. This keeps all content items from the same client in only one side of the evaluation, rather than allowing pages from one client to appear in both training and testing.

This is more honest for the question I am asking because the goal is to see whether the observed signal can generalize across clients. A random row-level split could be optimistic when multiple content items belong to the same client.

I will use `GroupShuffleSplit` with a 25% test size and `random_state=42`. The test set will therefore represent held-out clients rather than randomly selected content items.


In [3]:
# ML-08 — Connect to the FlyRank full-release warehouse

%pip -q install duckdb huggingface_hub

import os
import getpass

# Get Hugging Face token safely.
# Do NOT write the token directly into this notebook.
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

# Connect DuckDB.
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Hugging Face warehouse location.
REL = "hf://datasets/FlyRank/internship-warehouse"

# Warehouse tables.
TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Quick connection check.
print("DuckDB connected successfully.")
print("\nAvailable warehouse tables:")

for name in TABLES:
    print("-", name)

Paste your Hugging Face READ token (hf_...): ··········
DuckDB connected successfully.

Available warehouse tables:
- dim_clients
- dim_content
- fact_daily
- fact_daily_sample
- fact_query_90d


In [4]:

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit

# ---------------------------------------------------------
# 1. Build a 90-day feature table
# ---------------------------------------------------------

features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),

    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            -- Most recent 30 days
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            -- Previous 60 days
            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev60,

            -- Recent clicks
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            -- Average position in recent period
            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_last30,

            -- Position volatility: standard deviation
            STDDEV_SAMP(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_volatility

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY
            f.client_hash_id,
            f.content_hash_id

        HAVING imp_prev60 >= 100
    )

    SELECT *
    FROM windowed
""").df()

print(f"Feature rows: {len(features):,}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 136,252


,client_hash_id,content_hash_id,imp_last30,imp_prev60,clk_last30,pos_last30,pos_volatility
0,client_e547b89c05043229,content_1557a3abbc832229,198.0,302.0,2.0,11.738551,11.225065
1,client_e547b89c05043229,content_e1f6d0c859ba9dc4,195.0,554.0,0.0,23.401005,9.363426
2,client_e547b89c05043229,content_48537762b74f5b34,208.0,543.0,0.0,28.054512,17.485091
3,client_e547b89c05043229,content_27b27b5e13d4e6b7,134.0,424.0,1.0,24.978010,8.958257
4,client_e547b89c05043229,content_8c2c3dab1f1e875f,379.0,1459.0,0.0,34.729708,13.664352


In [5]:
# ---------------------------------------------------------
# 2. Add query-level signals
# ---------------------------------------------------------

qsignals = con.sql(f"""
    SELECT
        content_hash_id,

        ANY_VALUE(content_visible_query_count) AS visible_queries,

        ANY_VALUE(rare_impressions_share) AS rare_share,

        ANY_VALUE(anonymized_impressions_share) AS anon_share,

        MAX(impressions_90d) AS top_query_impressions,

        SUM(impressions_90d) AS kept_impressions

    FROM {TABLES['fact_query_90d']}

    GROUP BY content_hash_id
""").df()

qsignals["top_query_share"] = (
    qsignals["top_query_impressions"]
    / qsignals["kept_impressions"]
)

# Merge daily features with query-level features.
data = features.merge(
    qsignals,
    on="content_hash_id",
    how="left"
)

# ---------------------------------------------------------
# 3. Create the target
# ---------------------------------------------------------

# A decline means last-30-day impressions are below
# 80% of the previous-period impressions.
data["is_declining"] = (
    data["imp_last30"] < 0.8 * data["imp_prev60"]
).astype(int)

# ---------------------------------------------------------
# 4. Define the modeling features
# ---------------------------------------------------------

feature_cols = [
    "imp_prev60",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
    "pos_volatility"
]

# Keep rows where all model features are available.
model_data = data.dropna(
    subset=feature_cols + ["client_hash_id", "is_declining"]
).copy()

X = model_data[feature_cols]
y = model_data["is_declining"]
groups = model_data["client_hash_id"]

print(f"Modeling rows: {len(model_data):,}")
print(f"Number of clients: {groups.nunique():,}")
print(f"Declining rate: {y.mean():.3f}")

# ---------------------------------------------------------
# 5. Client-grouped train/test split
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_tr = X.iloc[train_idx]
X_te = X.iloc[test_idx]

y_tr = y.iloc[train_idx]
y_te = y.iloc[test_idx]

groups_tr = groups.iloc[train_idx]
groups_te = groups.iloc[test_idx]

# ---------------------------------------------------------
# 6. Honest split checks
# ---------------------------------------------------------

print("\n--- Split check ---")
print(f"Train rows: {len(X_tr):,}")
print(f"Test rows:  {len(X_te):,}")
print(f"Train clients: {groups_tr.nunique():,}")
print(f"Test clients:  {groups_te.nunique():,}")

shared_clients = set(groups_tr).intersection(set(groups_te))

print(f"Clients shared between train/test: {len(shared_clients)}")

print("\nTrain declining rate:", f"{y_tr.mean():.3f}")
print("Test declining rate: ", f"{y_te.mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 114,161
Number of clients: 48
Declining rate: 0.871

--- Split check ---
Train rows: 101,227
Test rows:  12,934
Train clients: 36
Test clients:  12
Clients shared between train/test: 0

Train declining rate: 0.901
Test declining rate:  0.630


## 2. Split design

I will train the shallow Decision Tree on the training clients and evaluate it only on the held-out clients.

For the comparison, I will use the same binary target and evaluate both the majority-class baseline and the Decision Tree on the same test set. I will report accuracy, precision, recall, and F1 so that the model is not judged by accuracy alone, especially because the target is imbalanced.

The baseline will always predict the majority class observed in the training data. This gives a simple reference point for deciding whether the Decision Tree provides useful predictive signal.


In [6]:
# ML-08 — Train Decision Tree and compare with majority baseline

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ---------------------------------------------------------
# 1. Majority-class baseline
# ---------------------------------------------------------

majority_class = y_tr.mode()[0]

baseline_pred = np.full(
    len(y_te),
    majority_class
)

# ---------------------------------------------------------
# 2. Train shallow Decision Tree
# ---------------------------------------------------------

tree = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=100,
    random_state=42
)

tree.fit(X_tr, y_tr)

tree_pred = tree.predict(X_te)

# ---------------------------------------------------------
# 3. Evaluation helper
# ---------------------------------------------------------

def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        )
    }

# ---------------------------------------------------------
# 4. Compare baseline vs Decision Tree
# ---------------------------------------------------------

results = pd.DataFrame([
    evaluate_model(
        "Week-4 majority baseline",
        y_te,
        baseline_pred
    ),
    evaluate_model(
        "ML-08 Decision Tree",
        y_te,
        tree_pred
    )
])

results.round(3)

,Model,Accuracy,Precision,Recall,F1
0,Week-4 majority baseline,0.630,0.630,1.000,0.773
1,ML-08 Decision Tree,0.683,0.668,0.991,0.798


In [7]:
# Confirm the baseline prediction and test distribution.

print("Majority class in training:", majority_class)
print("Actual test declining rate:", round(y_te.mean(), 3))

print("\nPrediction counts:")
print(pd.Series(tree_pred).value_counts().sort_index())

Majority class in training: 1
Actual test declining rate: 0.63

Prediction counts:
0      836
1    12098
Name: count, dtype: int64


In [8]:
print(results.round(3).to_string(index=False))

                   Model  Accuracy  Precision  Recall    F1
Week-4 majority baseline     0.630      0.630   1.000 0.773
     ML-08 Decision Tree     0.683      0.668   0.991 0.798


## 3. Train + compare vs my baseline

The Decision Tree improved on the majority baseline on the held-out clients. Its accuracy increased from 0.630 to 0.683, while F1 increased from 0.773 to 0.798.

The model still predicts the declining class for most test examples, which is consistent with the high declining rate in the test set. Its recall is 0.991, so it misses relatively few actual declining cases. The lower precision shows that some items predicted as declining did not actually decline.

I will inspect the confusion matrix and feature importance to understand these errors. Because this is a shallow tree, the feature importance should provide a directional view of which signals the model used most, rather than proving that any feature causes a decline.


In [9]:
from sklearn.metrics import confusion_matrix, classification_report

# ---------------------------------------------------------
# 1. Confusion matrix
# ---------------------------------------------------------

cm = confusion_matrix(y_te, tree_pred)

print("Confusion matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nError summary:")
print("True negatives :", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives  :", tp)

# ---------------------------------------------------------
# 2. Classification report
# ---------------------------------------------------------

print("\nClassification report:")
print(
    classification_report(
        y_te,
        tree_pred,
        target_names=["Not declining", "Declining"],
        zero_division=0
    )
)

# ---------------------------------------------------------
# 3. Feature importance
# ---------------------------------------------------------

importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": tree.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

print("\nFeature importance:")
print(importance.round(3).to_string(index=False))

Confusion matrix:
[[ 761 4019]
 [  75 8079]]

Error summary:
True negatives : 761
False positives: 4019
False negatives: 75
True positives  : 8079

Classification report:
               precision    recall  f1-score   support

Not declining       0.91      0.16      0.27      4780
    Declining       0.67      0.99      0.80      8154

     accuracy                           0.68     12934
    macro avg       0.79      0.58      0.53     12934
 weighted avg       0.76      0.68      0.60     12934


Feature importance:
        Feature  Importance
     imp_prev60       0.504
 pos_volatility       0.385
visible_queries       0.110
     rare_share       0.000
     anon_share       0.000
top_query_share       0.000


## 4. Errors and interpretation

The main observed error is false positives: the model predicted decline for 4,019 items that were not actually declining. In contrast, it produced only 75 false negatives, so it missed relatively few actual declining items.

This pattern explains the model's high recall for the declining class (0.991) but weak recall for the non-declining class (0.16). The model is therefore useful for catching potential declines, but its predictions should not be treated as definitive classifications.

The feature importance is concentrated in three variables. `imp_prev60` contributed 0.504 of the tree's importance, `pos_volatility` contributed 0.385, and `visible_queries` contributed 0.110. The other three features had zero importance in this fitted tree. These results are directional: they show which features the tree used for its decisions, not that these features cause impressions to decline.

Overall, the Decision Tree improved the observed F1 score from 0.773 for the majority baseline to 0.798, but the large number of false positives shows that the model still has limitations. I would treat it as decision-support for identifying content that may need attention rather than as a definitive prediction system.


In [10]:
print("=== ML-08 FINAL SUMMARY ===")
print(f"Baseline F1:    {results.loc[0, 'F1']:.3f}")
print(f"Decision Tree F1: {results.loc[1, 'F1']:.3f}")
print(f"\nF1 improvement: "
      f"{results.loc[1, 'F1'] - results.loc[0, 'F1']:+.3f}")
print(f"\nFalse positives: {fp:,}")
print(f"False negatives: {fn:,}")
print("\nTop features:")
print(importance.head(3).round(3).to_string(index=False))

=== ML-08 FINAL SUMMARY ===
Baseline F1:    0.773
Decision Tree F1: 0.798

F1 improvement: +0.025

False positives: 4,019
False negatives: 75

Top features:
        Feature  Importance
     imp_prev60       0.504
 pos_volatility       0.385
visible_queries       0.110


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.